# 📈 Real-Time Stock News Producer (Kafka)

This Jupyter Notebook acts as the streaming data generator for our real-time financial sentiment analysis pipeline. It simulates realistic financial news events for major tech and market-moving tickers and publishes them to an Apache Kafka cluster running in KRaft mode.

---

### 🔍 What this Producer does:
1. Connects to the local Apache Kafka broker listening on `localhost:9092`.
2. Inspects Kafka cluster topics and creates the `stock-news` topic if it does not already exist.
3. Simulates realistic market headlines across major tickers (`AAPL`, `TSLA`, `NVDA`, `MSFT`, `AMZN`, `GOOGL`, `META`, `AMD`, `JPM`, `BAC`).
4. Generates structured JSON payloads containing:
   - **`ticker`**: Stock symbol (e.g., `AAPL`, `TSLA`, `NVDA`)
   - **`headline`**: Realistic financial news headline (earnings beats, AI product launches, regulatory checks, etc.)
   - **`timestamp`**: Standard ISO 8601 UTC timestamp string
5. Publishes a new message to the `stock-news` topic every **2 seconds** in an infinite loop.
6. Gracefully handles manual interruptions (`KeyboardInterrupt`), ensuring Kafka producer buffers are flushed and connections are cleanly terminated.

---

### 🚀 How to Run in JupyterLab:
1. Make sure your Docker Kafka container is active (`docker compose up -d`).
2. Verify that the notebook kernel is set to **`Python (Stock Sentiment)`** (or your active `.venv`).
3. Select the code cell below and click the **Play (Run)** button or press `Shift + Enter`.
4. Observe the live stream of generated stock news printed in the cell output.

---

### ⏹️ How to Stop Safely:
- Click the **Square Stop Button ("Interrupt the kernel")** on the JupyterLab toolbar.
- Or navigate to the top menu: **Kernel -> Interrupt Kernel**.
- The `try ... except KeyboardInterrupt` block will catch the interrupt signal, flush any pending messages, close the producer, and exit cleanly.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
import json
import time
import random
from datetime import datetime, timezone
from kafka import KafkaProducer, KafkaAdminClient
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError, KafkaError, BrokerNotAvailableError

# ==========================================
# Configuration
# ==========================================
BOOTSTRAP_SERVERS = 'localhost:9092'
TOPIC_NAME = 'stock-news'
PUBLISH_INTERVAL_SECONDS = 2
MAX_MESSAGES = int(os.getenv('MAX_PRODUCER_MESSAGES', '0'))

# Top market tickers
TICKERS = ["AAPL", "TSLA", "NVDA", "MSFT", "AMZN", "GOOGL", "META", "AMD", "JPM", "BAC"]

# Realistic financial news headlines representing positive, negative, and neutral market events
HEADLINE_TEMPLATES = [
    # Positive market events
    "{ticker} reports quarterly earnings and revenue beating Wall Street estimates by a wide margin.",
    "{ticker} unveils next-generation enterprise AI accelerator chips, boosting forward guidance.",
    "Analyst upgrades {ticker} to Strong Buy with an aggressive price target increase.",
    "{ticker} announces record quarterly free cash flow and a new $15 billion share buyback program.",
    "Major European regulatory body greenlights {ticker}'s landmark multi-billion dollar acquisition.",
    "{ticker} forms strategic cloud computing and robotics alliance with global automotive leaders.",
    "Investor sentiment surges as {ticker} expands gross margins ahead of market expectations.",
    "{ticker} raises full-year dividend payout following robust institutional customer adoption.",

    # Negative market events
    "{ticker} faces new antitrust investigation from regulators over alleged market monopolization.",
    "{ticker} cuts full-year revenue outlook citing prolonged semiconductor supply chain constraints.",
    "Disappointing quarterly results: {ticker} misses EPS forecast as operating expenses rise sharply.",
    "Credit rating agency downgrades outlook on {ticker} senior unsecured corporate bonds.",
    "{ticker} shares tumble following executive departures and product delivery delays.",
    "Wall Street firm cuts {ticker} price target due to softening consumer demand and macro headwinds.",
    "{ticker} announces mandatory global recall of 85,000 hardware units due to battery defect.",
    "{ticker} CFO warns of persistent currency fluctuations and rising geopolitical risks.",

    # Neutral / Corporate market events
    "{ticker} schedules its Q3 fiscal financial results conference call and webcast for next week.",
    "{ticker} concludes annual general shareholder meeting and ratifies board appointments.",
    "{ticker} executive leadership speaks at Goldman Sachs Global Technology & Media Conference.",
    "{ticker} consolidates regional distribution hubs to streamline supply chain logistics.",
    "{ticker} trades sideways in quiet trading session ahead of Federal Reserve interest rate decision.",
    "{ticker} submits mandatory Form 8-K disclosure with the Securities and Exchange Commission."
]

def ensure_kafka_topic(bootstrap_servers, topic_name):
    """Verify or create the Kafka topic using KafkaAdminClient."""
    print(f"[Producer] Checking Kafka broker at {bootstrap_servers}...")
    try:
        admin_client = KafkaAdminClient(
            bootstrap_servers=bootstrap_servers,
            client_id='stock_news_admin',
            request_timeout_ms=5000
        )
        existing_topics = admin_client.list_topics()
        if topic_name in existing_topics:
            print(f"[Admin] Topic '{topic_name}' already exists.")
        else:
            print(f"[Admin] Topic '{topic_name}' not found. Creating topic...")
            new_topic = NewTopic(name=topic_name, num_partitions=1, replication_factor=1)
            admin_client.create_topics(new_topics=[new_topic], validate_only=False)
            print(f"[Admin] Topic '{topic_name}' created successfully.")
        admin_client.close()
    except TopicAlreadyExistsError:
        print(f"[Admin] Topic '{topic_name}' already exists.")
    except Exception as e:
        print(f"[Admin] Topic check/creation status: {e}")

def run_producer():
    """Main publishing loop generating and streaming stock news."""
    # 1. Ensure topic existence
    ensure_kafka_topic(BOOTSTRAP_SERVERS, TOPIC_NAME)

    # 2. Instantiate KafkaProducer
    print(f"[Producer] Initializing KafkaProducer for {BOOTSTRAP_SERVERS}...")
    try:
        producer = KafkaProducer(
            bootstrap_servers=BOOTSTRAP_SERVERS,
            value_serializer=lambda v: json.dumps(v).encode('utf-8'),
            acks='all',
            retries=3
        )
        print(f"[Producer] Connected successfully. Publishing to topic '{TOPIC_NAME}' every {PUBLISH_INTERVAL_SECONDS}s.")
        print("=" * 80)
    except Exception as e:
        print(f"[Producer Error] Failed to connect to Kafka at {BOOTSTRAP_SERVERS}: {e}")
        print("Please verify that Kafka is running via 'docker compose up -d'.")
        return

    # 3. Publish loop
    message_count = 0
    try:
        while True:
            ticker = random.choice(TICKERS)
            headline = random.choice(HEADLINE_TEMPLATES).format(ticker=ticker)
            timestamp = datetime.now(timezone.utc).isoformat()

            payload = {
                "ticker": ticker,
                "headline": headline,
                "timestamp": timestamp
            }

            producer.send(TOPIC_NAME, value=payload)
            producer.flush()

            message_count += 1
            print(f"[{message_count:04d}] Sent -> {ticker}: \"{headline}\" | {timestamp}")

            if MAX_MESSAGES > 0 and message_count >= MAX_MESSAGES:
                print(f"[Producer] Completed {MAX_MESSAGES} messages. Exiting loop cleanly.")
                break

            time.sleep(PUBLISH_INTERVAL_SECONDS)

    except KeyboardInterrupt:
        print("\n[Producer] KeyboardInterrupt received. Stopping producer loop safely...")
    except Exception as err:
        print(f"\n[Producer Error] Unexpected exception: {err}")
    finally:
        print("[Producer] Flushing pending messages and closing Kafka producer...")
        producer.flush()
        producer.close()
        print("[Producer] Shutdown complete. Kafka producer safely closed.")

# Run the producer
run_producer()


[Producer] Checking Kafka broker at localhost:9092...
[Admin] Topic 'stock-news' already exists.
[Producer] Initializing KafkaProducer for localhost:9092...
[Producer] Connected successfully. Publishing to topic 'stock-news' every 2s.
[0001] Sent -> TSLA: "Credit rating agency downgrades outlook on TSLA senior unsecured corporate bonds." | 2026-09-16T07:09:43.261390+00:00
[0002] Sent -> TSLA: "TSLA forms strategic cloud computing and robotics alliance with global automotive leaders." | 2026-09-16T07:09:45.408496+00:00
[0003] Sent -> JPM: "Credit rating agency downgrades outlook on JPM senior unsecured corporate bonds." | 2026-09-16T07:09:47.420444+00:00
[0004] Sent -> MSFT: "MSFT reports quarterly earnings and revenue beating Wall Street estimates by a wide margin." | 2026-09-16T07:09:49.430929+00:00
[0005] Sent -> AMD: "AMD raises full-year dividend payout following robust institutional customer adoption." | 2026-09-16T07:09:51.464809+00:00
[0006] Sent -> AMD: "AMD cuts full-year reve